# Session 07 — Video Processing Laboratory (2026)
**Google Colab notebook with 3 real demos:** Smart Traffic Monitoring, Drone Human Detection, and Smart CCTV Analytics.

**Recommended use in class**
1. Start with **Traffic** (easiest to understand).
2. Continue with **Drone Human Detection** (most challenging).
3. Finish with **Smart CCTV** (most relatable to daily life).

This notebook is designed for **Google Colab**. Core labs run on CPU. Optional YOLO sections may work better with a GPU.

## Learning outcomes
After completing the notebook, students should be able to:
- explain FPS, frame-based processing, detection, and tracking,
- run three real-world video analytics demos in Colab,
- compare classical OpenCV baselines with modern AI pipelines, and
- discuss accuracy, privacy, bias, and responsible AI.

## Setup — imports and helper functions

In [ ]:
import os, cv2, math, numpy as np, subprocess
from matplotlib import pyplot as plt
from IPython.display import Video, display, HTML

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

OUT = os.getcwd()

def path(name):
    return os.path.join(OUT, name)

def show_video(p, width=720):
    display(Video(p, embed=True, width=width, html_attributes='controls'))

def mp4_writer(p, fps, wh):
    return cv2.VideoWriter(p, cv2.VideoWriter_fourcc(*'mp4v'), fps, wh)

def make_browser_mp4(source):
    target = source.replace('.mp4','_web.mp4')
    try:
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',source,'-c:v','libx264','-pix_fmt','yuv420p',target], check=True)
        return target
    except Exception:
        return source

def upload_video():
    if not IN_COLAB:
        raise RuntimeError('This upload helper is intended for Google Colab.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    return next(iter(uploaded))

class CentroidTracker:
    def __init__(self, max_distance=55):
        self.max_distance = max_distance
        self.next_id = 1
        self.objects = {}
        self.history = {}
    def update(self, detections):
        # detections: list of (x,y,w,h)
        centers = [((x+w//2),(y+h//2),(x,y,w,h)) for (x,y,w,h) in detections]
        assigned = set()
        new_objects = {}
        new_history = {}
        for oid,(cx_old,cy_old,bbox_old) in self.objects.items():
            best = None; best_d = 1e9; best_idx = None
            for idx,(cx,cy,bbox) in enumerate(centers):
                if idx in assigned: continue
                d = math.hypot(cx-cx_old, cy-cy_old)
                if d < best_d:
                    best_d = d; best = (cx,cy,bbox); best_idx = idx
            if best is not None and best_d <= self.max_distance:
                assigned.add(best_idx)
                new_objects[oid] = best
                hist = self.history.get(oid, []) + [(best[0], best[1])]
                new_history[oid] = hist[-30:]
        for idx,(cx,cy,bbox) in enumerate(centers):
            if idx not in assigned:
                oid = self.next_id; self.next_id += 1
                new_objects[oid] = (cx,cy,bbox)
                new_history[oid] = [(cx,cy)]
        self.objects = new_objects
        self.history = new_history
        return self.objects, self.history

def draw_tracks(frame, tracker_objects, tracker_history, color=(0,255,0), label='ID'):
    for oid,(cx,cy,bbox) in tracker_objects.items():
        x,y,w,h = bbox
        cv2.rectangle(frame,(x,y),(x+w,y+h),color,2)
        cv2.putText(frame,f'{label} {oid}',(x,max(18,y-8)),cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
        pts = tracker_history.get(oid, [])
        for i in range(1,len(pts)):
            cv2.line(frame, pts[i-1], pts[i], color, 2)
    return frame


## Scenario map — which demo should students try first?
| Scenario | Best camera view | Main targets | Core tasks | Suggested first use |
|---|---|---|---|---|
| Smart Traffic Monitoring | Fixed roadside camera | Cars, buses, motorcycles | Detection, tracking, line counting | **Start here** |
| Drone Human Detection | Aerial drone video | People / hikers | Detection, search, path tracking | Advanced challenge |
| Smart CCTV Analytics | Static surveillance camera | Pedestrians | Detection, entry/exit, occupancy | Real-life campus example |

## Warm-up Lab — FPS and frames
This short lab keeps the notebook aligned with the lecture slides. Before using the three real demos, students should review how a video is built from frames.

In [ ]:
W,H,N,FPS = 640,360,120,24
raw = path('fps_demo.mp4')
wr = mp4_writer(raw,FPS,(W,H))
for i in range(N):
    frame = np.full((H,W,3),(30,32,38),dtype=np.uint8)
    for x in range(0,W,40): cv2.line(frame,(x,0),(x,H),(54,56,60),1)
    for y in range(0,H,40): cv2.line(frame,(0,y),(W,y),(54,56,60),1)
    cv2.rectangle(frame,(40+i*3,120),(120+i*3,180),(70,170,255),-1)
    cv2.circle(frame,(530-i*2,240),28,(70,220,110),-1)
    cv2.putText(frame,f'Frame {i:03d} | {FPS} FPS',(15,30),cv2.FONT_HERSHEY_SIMPLEX,0.75,(255,255,255),2)
    wr.write(frame)
wr.release()
show_video(make_browser_mp4(raw), 640)
print('Question: what changes if the exact same frames are played at 12 FPS or 48 FPS?')

# DEMO A — Smart Traffic Monitoring
Use a short roadside video with moving vehicles. This is the easiest scenario for students because the targets are larger and easier to interpret visually.

### Step A1 — Upload a traffic video
In Google Colab, run the next cell and upload a short `.mp4` clip.

In [ ]:
# traffic_input = upload_video()  # uncomment in Colab
traffic_input = 'traffic_demo.mp4'  # replace with your uploaded filename

### Step A2 — Process the traffic video with a CPU-friendly baseline
This baseline uses **background subtraction + contour detection + centroid tracking**. It is not as strong as YOLO, but it runs without paid software.

In [ ]:
def process_traffic(input_video, output_video='traffic_annotated.mp4', min_area=900, line_ratio=0.68):
    cap = cv2.VideoCapture(input_video)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError('Cannot read the traffic video. Check the filename.')
    H, W = frame.shape[:2]
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    line_y = int(H * line_ratio)
    writer = mp4_writer(output_video, fps, (W, H))
    fgbg = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=35, detectShadows=True)
    tracker = CentroidTracker(max_distance=60)
    crossed = set(); total = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        mask = fgbg.apply(frame)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
        kernel = np.ones((5,5), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_DILATE, kernel, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        detections = []
        for c in contours:
            area = cv2.contourArea(c)
            if area >= min_area:
                x,y,w,h = cv2.boundingRect(c)
                detections.append((x,y,w,h))
        objs, hist = tracker.update(detections)
        cv2.line(frame, (0,line_y), (W,line_y), (0,255,255), 2)
        for oid,(cx,cy,bbox) in objs.items():
            pts = hist.get(oid, [])
            if len(pts) >= 2:
                prev_y = pts[-2][1]
                if prev_y < line_y <= cy and oid not in crossed:
                    crossed.add(oid); total += 1
        draw_tracks(frame, objs, hist, color=(0,165,255), label='Vehicle')
        cv2.putText(frame, f'Count: {total}', (20,35), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        writer.write(frame)
    cap.release(); writer.release()
    return output_video, total

# out_traffic, traffic_count = process_traffic(traffic_input)
# show_video(make_browser_mp4(out_traffic), 720)
# print('Vehicles counted:', traffic_count)

**Discussion questions**
- Why can shadows and dense traffic reduce counting accuracy?
- Why is this scenario usually easier than drone human detection?
- Which metric is most useful here: count, speed, or track stability?

# DEMO B — Drone Human Detection
Use an aerial drone video over a trail, field, or outdoor area. This scenario connects strongly to mountain safety and search-and-rescue.

### Step B1 — Upload a drone video

In [ ]:
# drone_input = upload_video()  # uncomment in Colab
# drone_input = 'drone_demo.mp4'  # replace with your uploaded filename

### Step B2 — CPU baseline with HOG people detection
This baseline is intentionally simple. It may miss small people in aerial videos, which makes it a great discussion point for students.

In [ ]:
def process_drone_people(input_video, output_video='drone_people_annotated.mp4', scale_width=960):
    hog = cv2.HOGDescriptor()
    hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
    cap = cv2.VideoCapture(input_video)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError('Cannot read the drone video. Check the filename.')
    H0, W0 = frame.shape[:2]
    scale = scale_width / W0 if W0 > scale_width else 1.0
    W, H = int(W0 * scale), int(H0 * scale)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    writer = mp4_writer(output_video, fps, (W, H))
    tracker = CentroidTracker(max_distance=80)
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if scale != 1.0:
            frame = cv2.resize(frame, (W, H))
        rects, weights = hog.detectMultiScale(frame, winStride=(4,4), padding=(8,8), scale=1.05)
        detections = []
        for (x,y,w,h), conf in zip(rects, weights):
            if conf >= 0.2:
                detections.append((int(x),int(y),int(w),int(h)))
        objs, hist = tracker.update(detections)
        draw_tracks(frame, objs, hist, color=(0,255,0), label='Person')
        cv2.putText(frame, f'Frame: {frame_idx} | Detected: {len(objs)}', (18,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        cv2.putText(frame, 'CPU baseline: HOG people detector', (18,60), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
        writer.write(frame)
        frame_idx += 1
    cap.release(); writer.release()
    return output_video

# out_drone = process_drone_people(drone_input)
# show_video(make_browser_mp4(out_drone), 720)

### Optional stronger version — YOLO + tracking
If your Colab runtime has internet and a GPU, you can try a stronger pipeline.

In [ ]:
# OPTIONAL — uncomment in Colab when internet/model download is available
# %pip -q install ultralytics
# from ultralytics import YOLO
# model = YOLO('yolov8n.pt')
# result = model.track(source=drone_input, tracker='bytetrack.yaml', conf=0.25, classes=[0], save=True)
# print('Check the runs/track folder for the annotated output.')

**Discussion questions**
- Why are tiny objects from an aerial view hard to detect?
- How do camera altitude, angle, and vegetation affect accuracy?
- Why is a stronger model often needed here compared with traffic monitoring?

# DEMO C — Smart CCTV Analytics
Use a fixed surveillance video such as a gate, lobby, hallway, or public space. This scenario is ideal for discussing people counting, occupancy, and privacy.

### Step C1 — Upload a CCTV video

In [ ]:
# cctv_input = upload_video()  # uncomment in Colab
# cctv_input = 'cctv_demo.mp4'  # replace with your uploaded filename

### Step C2 — Count pedestrian flow with a CPU baseline
This baseline again uses **background subtraction + centroid tracking**, but the task now focuses on entry/exit flow and occupancy.

In [ ]:
def process_cctv(input_video, output_video='cctv_annotated.mp4', min_area=550, line_ratio=0.55):
    cap = cv2.VideoCapture(input_video)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError('Cannot read the CCTV video. Check the filename.')
    H, W = frame.shape[:2]
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    line_y = int(H * line_ratio)
    writer = mp4_writer(output_video, fps, (W, H))
    fgbg = cv2.createBackgroundSubtractorMOG2(history=250, varThreshold=28, detectShadows=True)
    tracker = CentroidTracker(max_distance=45)
    entered, exited = set(), set()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        mask = fgbg.apply(frame)
        _, mask = cv2.threshold(mask, 200, 255, cv2.THRESH_BINARY)
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.dilate(mask, kernel, iterations=2)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        detections = []
        for c in contours:
            area = cv2.contourArea(c)
            if area >= min_area:
                x,y,w,h = cv2.boundingRect(c)
                # filter out very wide blobs likely not pedestrian-like
                if h > 1.2*w:
                    detections.append((x,y,w,h))
        objs, hist = tracker.update(detections)
        cv2.line(frame, (0,line_y), (W,line_y), (255,255,0), 2)
        for oid,(cx,cy,bbox) in objs.items():
            pts = hist.get(oid, [])
            if len(pts) >= 2:
                prev_y = pts[-2][1]
                if prev_y < line_y <= cy:
                    entered.add(oid)
                elif prev_y > line_y >= cy:
                    exited.add(oid)
        draw_tracks(frame, objs, hist, color=(180,120,255), label='Ped')
        cv2.putText(frame, f'Entered: {len(entered)}  Exited: {len(exited)}', (18,32), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 2)
        cv2.putText(frame, f'Occupancy estimate: {max(0, len(entered)-len(exited))}', (18,64), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255,255,255), 2)
        writer.write(frame)
    cap.release(); writer.release()
    return output_video

# out_cctv = process_cctv(cctv_input)
# show_video(make_browser_mp4(out_cctv), 720)

**Discussion questions**
- Why is a fixed camera helpful here?
- Which matters more for this use case: total count or occupancy over time?
- What privacy safeguards should be applied on a campus or office CCTV system?

## Mini Lab — Video restoration (optional)
This section keeps the notebook aligned with the lecture slide on restoration.

In [ ]:
# To test restoration, capture one frame from any processed output and degrade it artificially.
# Example: use the traffic video output after you have run the earlier cell.
# sample_frame_source = out_traffic
# cap = cv2.VideoCapture(sample_frame_source)
# ok, frame = cap.read(); cap.release()
# if ok:
#     noise = np.random.normal(0, 28, frame.shape)
#     degraded = np.clip(frame.astype(np.float32)*0.65 + noise, 0, 255).astype(np.uint8)
#     denoised = cv2.fastNlMeansDenoisingColored(degraded, None, 8, 8, 7, 21)
#     sharpen = cv2.addWeighted(denoised, 1.4, cv2.GaussianBlur(denoised, (0,0), 1.2), -0.4, 0)
#     fig, axes = plt.subplots(1,4,figsize=(16,4))
#     for ax,img,title in zip(axes,[frame,degraded,denoised,sharpen],['Original','Degraded','Denoised','Sharpened']):
#         ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax.set_title(title); ax.axis('off')
#     plt.tight_layout(); plt.show()

## Concept Lab — Motion and 3D inference
Important reminder: **tracking motion is not the same as reconstructing 3D geometry**.

In [ ]:
# Example idea:
# - choose two frames from traffic or CCTV output
# - compute optical flow
# - discuss why image motion does not directly tell us real-world depth in meters
# The instructor can keep this section conceptual if time is short.

## Closing comparison activity
Ask students to compare the three demos.

| Question | Traffic | Drone | CCTV |
|---|---|---|---|
| Easiest targets to see? | Usually yes | Often no | Moderate |
| Camera motion? | Usually fixed | Often moving | Fixed |
| Best classical baseline? | Background subtraction | HOG / weak baseline | Background subtraction |
| Most demanding ethics discussion? | Moderate | Search-and-rescue context | **High** (privacy) |

### Instructor answer key
- **Traffic** is the easiest starting point because objects are large and camera geometry is stable.
- **Drone** is the hardest because people are small and the viewpoint changes.
- **CCTV** is the best scenario for discussing occupancy, entry/exit flow, and privacy safeguards.

### Suggested homework
Each group chooses **one** of the three demos and writes:
1. the input video type,
2. the chosen pipeline,
3. one likely failure case,
4. one improvement idea, and
5. one responsible-AI consideration.